# Extra · A Hovmöller diagram of the West African monsoon

A **Hovmöller diagram** shows a value against **position** and **time** in one picture: here latitude
against month. It shows things that move, like the rain band of the West African monsoon, which travels north
in summer and brings the rain to the Sahel ([Da-Allada et al., 2020](https://agupubs.onlinelibrary.wiley.com/doi/10.1029/2020EF001595)).
In this notebook you

1. select the cells of a region, between 10°W and 10°E, from 10°S to 20°N,
2. average the precipitation of each month over the years and over each **latitude**,
3. draw the Hovmöller diagram for 1950–1980 and 1981–2023, and their difference.

Data: ERA5 precipitation, monthly, at zoom 7 (~50 km).

## 1. The cells of the region

As in the extra *Regional analysis*: the cells whose position is inside the limits. They are few, so
`.compute()` loads them into memory: grouping by latitude needs it.

In [ ]:
import intake
import matplotlib.pyplot as plt
import numpy as np

cat = intake.open_catalog("https://data.nextgems-h2020.eu/online.yaml")
era5 = cat.ERA5(zoom=7).to_dask()

lon = (era5["lon"].values + 180) % 360 - 180  # longitudes from -180 to 180
lat = era5["lat"].values
inside = (lon > -10) & (lon < 10) & (lat > -10) & (lat < 20)
print(f"{inside.sum():,} cells")

precipitation = (
    era5["tp"].isel(cell=np.where(inside)[0]).compute()
)  # into memory: about 10 MB
precipitation.attrs

ERA5 gives the precipitation in **m per day** (the mean of the days of the month); × 1000 gives mm per day.

## 2. Month and latitude

Two means, one after the other:

- `.groupby("time.month").mean()`: the mean of each month over the years of a period,
- `.groupby("lat").mean()`: the mean of all cells with the same latitude. HEALPix cells lie on rings of equal
  latitude, so every ring gives one value: the mean between 10°W and 10°E.

In [ ]:
def hovmoller(start, end):
    period = precipitation.sel(time=slice(start, end))
    return 1000 * period.groupby("time.month").mean().groupby("lat").mean()


reference = hovmoller("1950", "1980")
recent = hovmoller("1981", "2023")
reference

## 3. The diagrams

Month on the horizontal axis, latitude on the vertical. Dashed lines: 5°N, about the coast of the Gulf of
Guinea, and 10°N, where the rain of the monsoon reaches the Sahel.

In [ ]:
month_names = ["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, data, title in [
    (axes[0], reference, "1950–1980"),
    (axes[1], recent, "1981–2023"),
]:
    data.plot(
        ax=ax,
        x="month",
        y="lat",
        cmap="YlGnBu",
        vmin=0,
        vmax=10,
        cbar_kwargs={"label": "precipitation / mm per day"},
    )
    ax.set_title(title)
for ax in axes:
    ax.axhline(5, color="black", linestyle="--")
    ax.axhline(10, color="black", linestyle="--")
    ax.set_xticks(range(1, 13), month_names)
    ax.set_xlabel("month")
axes[0].set_ylabel("latitude / °N")
axes[1].set_ylabel("")

fig.savefig("/app/pages/hovmuller_diagram/hovmuller.png", dpi=100, bbox_inches="tight")
plt.show()

Follow the band of rain: at the coast (5°N) in May and June, at 10°N and further north in August, and
back south in autumn.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
(recent - reference).plot(
    ax=ax,
    x="month",
    y="lat",
    cmap="BrBG",
    vmin=-2,
    vmax=2,
    cbar_kwargs={"label": "1981–2023 − 1950–1980 / mm per day"},
)
ax.axhline(5, color="black", linestyle="--")
ax.axhline(10, color="black", linestyle="--")
ax.set_xticks(range(1, 13), month_names)
ax.set_xlabel("month")
ax.set_ylabel("latitude / °N")
ax.set_title("Change of the West African monsoon rain")

fig.savefig(
    "/app/pages/hovmuller_diagram/hovmuller_difference.png",
    dpi=100,
    bbox_inches="tight",
)
plt.show()

## 5. Exercise

A Hovmöller diagram of another monsoon: the **Indian monsoon**, e.g. 70°E to 90°E and 10°S to 30°N. When does
its rain arrive at 20°N? Save your diagram as `/app/pages/hovmuller_diagram/hovmuller_exercise.png`.
(Copy section 1 with the new limits, then `hovmoller` and the plot.)

In [ ]:
# the Indian monsoon

## Your page

Open `pages/hovmuller_diagram/my_page.rst` in JupyterLab and describe what you found. Look at it on
http://localhost:8000 → *Extras*.